In [ ]:
# -- Cell 0: Pre-flight ----------------------------------------------------
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# Import datasets and trl BEFORE torch initialises the CUDA context.
# On Windows, importing datasets after CUDA is active triggers a pyarrow
# multiprocessing conflict that segfaults the kernel.
import datasets
import trl
import subprocess
import torch

assert torch.cuda.is_available(), 'CUDA not available -- run finetuning/test_env.py.'

total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'Device: {torch.cuda.get_device_name(0)}  |  Total VRAM: {total_gb:.1f} GB')
try:
    smi = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=memory.used,memory.free', '--format=csv,noheader,nounits'],
        text=True,
    ).strip()
    used_mb, free_mb = [int(x.strip()) for x in smi.split(',')]
    print(f'nvidia-smi -- Used: {used_mb} MiB  |  Free: {free_mb} MiB')
    if free_mb < 7000:
        print(f'\nWARN: Only {free_mb} MiB free. Close other GPU apps before proceeding.')
    else:
        print(f'\nOK: {free_mb} MiB available. Safe to proceed.')
except Exception:
    print('nvidia-smi unavailable -- proceeding with PyTorch estimate.')

print(f"\ntransformers : {__import__('transformers').__version__}")
print(f'datasets     : {datasets.__version__}')
print(f'trl          : {trl.__version__}')


In [ ]:
# -- Cell 1: Safe imports ---------------------------------------------------
import os, gc, json, re
from collections import Counter
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, TaskType, get_peft_model, PeftModel
from sklearn.model_selection import train_test_split

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('Safe imports OK.')


## Section 1 -- Configuration

Paths to all adapter directories. ADP-C v1 is the inference oracle; ADP-C v2
is the retrained version produced by this step. v1 is preserved at
`adp_c_final/` for rollback and for re-runs of Steps 13 and 15.


In [ ]:
# -- Cell 2: Configuration --------------------------------------------------
GEMMA_BASE_ID = 'google/gemma-2-2b-it'
QWEN_BASE_ID  = 'Qwen/Qwen3-4B'

BASE_DIR   = Path(r'D:/Git Repos/nikko-companion')
FINETUNING = BASE_DIR / 'finetuning'
ADP_B_DIR  = FINETUNING / 'adp_b_safety'    / 'adp_b_final'
ADP_C_V1   = FINETUNING / 'adp_c_evaluator' / 'adp_c_final'
ADP_C_V2   = FINETUNING / 'adp_c_evaluator' / 'adp_c_v2_final'

for d in [ADP_B_DIR, ADP_C_V1]:
    assert d.exists(), f'Required adapter not found: {d}'
ADP_C_V2.mkdir(parents=True, exist_ok=True)

# -- Training hyperparameters ------------------------------------------------
N_SAMPLES     = 120    # ADP-A responses to generate for v2 training data
NUM_EPOCHS    = 10     # cosine LR + best-model checkpoint prevents overfit
BATCH_SIZE    = 4      # per-device train batch size
GRAD_ACCUM    = 8      # effective batch = 4 x 8 = 32
LEARNING_RATE = 1e-4
WEIGHT_DECAY  = 0.01
LORA_R        = 8
LORA_ALPHA    = 16
LORA_DROPOUT  = 0.05
LORA_TARGETS  = ['q_proj', 'k_proj', 'v_proj', 'o_proj']
# LORA_TARGETS: the four attention projection matrices. Adapting q/k/v/o
# captures task-specific attention behaviour (verdict generation) without
# adapting the heavier MLP layers.

# -- Generation hyperparameters (Phase A) ------------------------------------
GEN_BATCH_SIZE = 4
# GEN_BATCH_SIZE: prompts processed in a single .generate() call.
# When Qwen3-4B layers are partially CPU-offloaded (free VRAM < ~8 GB),
# each .generate() call incurs one PCIe round-trip per forward pass.
# Batching GEN_BATCH_SIZE prompts reduces total round-trips by GEN_BATCH_SIZE x
# while the GPU processes all sequences in parallel during on-GPU passes.
# Reduce to 2 if you hit OOM (KV cache scales linearly with batch size).

MAX_NEW_TOKENS = 128
# MAX_NEW_TOKENS: cap on generated response length. 128 tokens ~ 100 words;
# sufficient for concise Nikko support responses. Reduced from 200 to cut
# per-sample wall-time by ~36% (long-tail tokens past ~120 are typically
# repetitive for short empathic responses).

print(f'ADP-B (Gemma-2) : {ADP_B_DIR}')
print(f'ADP-C v1 oracle : {ADP_C_V1}')
print(f'ADP-C v2 output : {ADP_C_V2}')
print(f'N_SAMPLES={N_SAMPLES}  EPOCHS={NUM_EPOCHS}  eff_batch={BATCH_SIZE*GRAD_ACCUM}')
print(f'GEN_BATCH_SIZE={GEN_BATCH_SIZE}  MAX_NEW_TOKENS={MAX_NEW_TOKENS}')


## Section 2 -- Phase A: Load Qwen3-4B and Generate Samples

**Restructured load order:**

| Original | Problem | New order |
|---|---|---|
| Gemma-2 first (4.95 GB) -> Qwen3 | Only ~3 GB VRAM left -> CPU offload -> ~20x slower | Qwen3 first (full 8 GB) -> generate -> free -> Gemma-2 |

With a clean VRAM slate, `max_memory={0: '7500MiB'}` keeps all Qwen3 layers
on cuda:0. After generation, Qwen3 is fully freed before Gemma-2 loads.


In [ ]:
# -- Cell 3: Load Qwen3-4B (ADP-A oracle) -----------------------------------
# [CONCEPT] Loading Qwen3 FIRST is the key restructure. With a clean VRAM
# slate (~8 GB free), device_map='auto' places all layers on cuda:0.
# After generation they are freed entirely before Gemma-2 loads, so the
# two models never compete for VRAM simultaneously.
print(f'Loading Qwen3-4B base ({QWEN_BASE_ID})...')
qwen_tok = AutoTokenizer.from_pretrained(QWEN_BASE_ID)
if qwen_tok.pad_token is None: qwen_tok.pad_token = qwen_tok.eos_token

qwen_base = AutoModelForCausalLM.from_pretrained(
    QWEN_BASE_ID,
    torch_dtype=torch.bfloat16,
    # Cap GPU at 7.5 GB -- leaves 0.5 GB headroom for KV cache and activations.
    # Qwen3-4B is ~8 GB in bf16; with a clean VRAM slate, all layers fit on GPU.
    max_memory={0: '7500MiB', 'cpu': '16GiB'},
    device_map='auto',
)
adp_a_model = qwen_base
adp_a_model.eval()
print(f'ADP-A (Qwen3-4B) loaded. VRAM: {torch.cuda.memory_allocated(0)/1024**3:.2f} GB')


In [ ]:
# -- Cell 4: Generate ADP-A pipeline samples (batched) ----------------------
#
# WHAT CHANGED vs. the original:
#   Original: run_adp_a() called 120 times in a loop -- one .generate() per sample.
#   When Qwen3-4B is partially CPU-offloaded (Cell 3 warning), each call incurs
#   a full PCIe round-trip for offloaded layers: 120 samples = 120x transfers.
#
#   New: run_adp_a_batch() processes GEN_BATCH_SIZE prompts per .generate() call,
#   reducing PCIe-bound passes by GEN_BATCH_SIZE. Also: max_new_tokens 200->128.

SYSTEM_PREAMBLE = (
    'You are Nikko, a non-clinical AI mental health companion. '
    'You provide emotional support and evidence-based information. '
    'You are NOT a therapist, psychologist, or medical professional. '
    'You MUST NOT diagnose, prescribe, or replace professional care.'
)

SAMPLE_PROMPTS = [
    "I've been feeling really anxious lately and I'm not sure why.",
    "I don't know how to cope with my grief after losing my dad.",
    "Everything feels pointless. I can't find motivation for anything.",
    "I feel like no one understands what I'm going through.",
    "I've been crying a lot and I don't know what to do.",
    "I'm exhausted all the time, both mentally and physically.",
    "I feel disconnected from the people around me.",
    "I've been struggling with negative thoughts about myself.",
    "I feel overwhelmed by everything in my life right now.",
    "I don't enjoy things I used to love anymore.",
    "I've been having panic attacks and they scare me.",
    "I feel like a burden to the people who care about me.",
    "I've been isolating myself more and more.",
    "I'm scared about the future and what it holds.",
    "I feel empty inside, like something is missing.",
] * (N_SAMPLES // 15 + 1)
SAMPLE_PROMPTS = SAMPLE_PROMPTS[:N_SAMPLES]

# [CONCEPT] Left-padding for batched decoder-model generation:
# Decoder-only models generate left-to-right. In a batch, all sequences must
# be the same length, so shorter ones are padded. The padding MUST go on the
# LEFT so all real content is flush at the right edge -- this is where the
# model continues generation. Right-padding causes generation after pad tokens,
# which is undefined behaviour and produces garbage for chat-tuned models.
qwen_tok.padding_side = 'left'


def run_adp_a_batch(prompts):
    """Generate Nikko responses for a list of prompts in one batched .generate() call.

    The GPU processes all sequences in parallel during the on-GPU portion of
    each forward pass. With partial CPU offload, this reduces total PCIe
    round-trips by GEN_BATCH_SIZE compared to sequential single-sample calls.
    """
    # Format each prompt into Qwen3's chat template.
    # enable_thinking=False: suppresses Qwen3's <think>...</think> reasoning
    # traces -- we want direct support responses, not chain-of-thought tokens.
    formatted = [
        qwen_tok.apply_chat_template(
            [
                {'role': 'system', 'content': SYSTEM_PREAMBLE},
                {'role': 'user',   'content': p},
            ],
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
        for p in prompts
    ]

    # Tokenize the full batch with left-padding.
    # max_length=400 caps the prompt so total sequence length stays within
    # 400+128=528 tokens. Prompts here are ~50 tokens, so truncation is a
    # safety net only.
    inputs = qwen_tok(
        formatted,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=400,
    )

    # Route to the device holding the first model layer. With partial CPU
    # offload, device_map='auto' places early layers on cuda:0. Accelerate's
    # hooks handle inter-device transfers during the forward pass.
    first_device = next(adp_a_model.parameters()).device
    inputs = {k: v.to(first_device) for k, v in inputs.items()}

    with torch.no_grad():
        # torch.no_grad(): skips gradient tracking -- inference only.
        # Saves ~30% memory and skips gradient-tape overhead per forward pass.
        outputs = adp_a_model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=0.75,
            top_p=0.92,
            repetition_penalty=1.1,
            pad_token_id=qwen_tok.eos_token_id,
        )

    # outputs shape: (batch_size, padded_input_len + MAX_NEW_TOKENS).
    # inputs['input_ids'].shape[1] is the uniform padded input length.
    # Slicing from that index strips the prompt prefix for every item.
    prompt_len = inputs['input_ids'].shape[1]
    return [
        qwen_tok.decode(out[prompt_len:], skip_special_tokens=True).strip()
        for out in outputs
    ]


import time
print(f'Generating {N_SAMPLES} samples (batch={GEN_BATCH_SIZE}, max_new_tokens={MAX_NEW_TOKENS}).')
print('Tip: if Cell 3 warned about CPU offload, close Chrome/Discord and re-run from Cell 3.')
print()

t0 = time.time()
samples = []

for batch_start in range(0, N_SAMPLES, GEN_BATCH_SIZE):
    batch     = SAMPLE_PROMPTS[batch_start : batch_start + GEN_BATCH_SIZE]
    responses = run_adp_a_batch(batch)
    for prompt, response in zip(batch, responses):
        samples.append({'user': prompt, 'response': response})
    completed = min(batch_start + GEN_BATCH_SIZE, N_SAMPLES)
    if completed % 20 == 0 or completed == N_SAMPLES:
        elapsed = time.time() - t0
        rate    = completed / elapsed if elapsed > 0 else 0
        eta     = (N_SAMPLES - completed) / rate if rate > 0 else 0
        print(f'  {completed}/{N_SAMPLES}  |  {elapsed:.0f}s elapsed  |  ETA {eta:.0f}s')

total = time.time() - t0
print(f'Generated {len(samples)} samples in {total:.0f}s ({total/N_SAMPLES:.1f}s per sample).')

# -- Free Qwen3-4B before loading Gemma-2 -------------------------------------
# del removes Python references; gc.collect() reclaims CPU RAM;
# torch.cuda.empty_cache() releases CUDA allocator cached blocks back to the
# driver -- without it, freed GPU memory stays in PyTorch's pool and appears
# 'used' to the OS, blocking the Gemma-2 load.
del adp_a_model, qwen_base, qwen_tok
gc.collect()
torch.cuda.empty_cache()
print(f'VRAM after Qwen3-4B release: {torch.cuda.memory_allocated(0)/1024**3:.2f} GB')
print('Expected: < 0.5 GB. Safe to load Gemma-2.')


## Section 3 -- Phase B: Score Samples with ADP-B + ADP-C v1

Load Gemma-2 once as the shared base. Both ADP-B and ADP-C v1 are mounted as
named adapters and hot-swapped via `set_adapter()` -- no base reload needed.

| Pass | Adapter | Purpose |
|---|---|---|
| 1 | `adp_b` | Crisis flag classification |
| 2 | `adp_c_v1` | Response verdict (PASS / REGENERATE / ESCALATE) |


In [ ]:
# -- Cell 5: Load Gemma-2 + ADP-B + ADP-C v1 --------------------------------
# [CONCEPT] PeftModel supports multiple named adapters on a single frozen base.
# Loading both ADP-B and ADP-C v1 and calling set_adapter() to switch between
# them avoids reloading the 4 GB base between scoring passes.
print(f'Loading Gemma-2 base ({GEMMA_BASE_ID})...')
gemma_tok = AutoTokenizer.from_pretrained(GEMMA_BASE_ID)
if gemma_tok.pad_token is None: gemma_tok.pad_token = gemma_tok.eos_token

gemma_base = AutoModelForCausalLM.from_pretrained(
    GEMMA_BASE_ID, torch_dtype=torch.bfloat16, device_map='auto'
)
# Load ADP-B as primary, then add ADP-C v1 as a second named adapter.
adp_bc_model = PeftModel.from_pretrained(gemma_base, str(ADP_B_DIR), adapter_name='adp_b')
adp_bc_model.load_adapter(str(ADP_C_V1), adapter_name='adp_c_v1')
adp_bc_model.eval()
print(f'Gemma-2 + ADP-B + ADP-C v1 loaded. VRAM: {torch.cuda.memory_allocated(0)/1024**3:.2f} GB')


In [ ]:
# -- Cell 6: Score samples with ADP-B + ADP-C v1 ----------------------------

def _gemma_generate(prompt_text, max_new=120, greedy=True):
    # Shared generation helper -- avoids duplicating the tokenise/decode boilerplate.
    inputs = gemma_tok(prompt_text, return_tensors='pt').to(gemma_base.device)
    kwargs = dict(max_new_tokens=max_new, pad_token_id=gemma_tok.eos_token_id)
    if not greedy:
        kwargs['do_sample'] = True
    else:
        kwargs['do_sample'] = False
    with torch.no_grad():
        out = adp_bc_model.generate(**inputs, **kwargs)
    return gemma_tok.decode(out[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True).strip()


def run_adp_b(user_text):
    adp_bc_model.set_adapter('adp_b')
    msgs = [{'role': 'user', 'content': f'[SAFETY ASSESSMENT]\n{user_text}'}]
    prompt = gemma_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    return _gemma_generate(prompt, max_new=100)


def run_adp_c_v1(user_text, response):
    adp_bc_model.set_adapter('adp_c_v1')
    content = f'[USER]: {user_text}\n[NIKKO RESPONSE]: {response}'
    msgs = [{'role': 'user', 'content': content}]
    prompt = gemma_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    return _gemma_generate(prompt, max_new=150)


def parse_verdict(raw):
    # Extract verdict from JSON output; fall back to keyword scan, then PASS.
    try:
        match = re.search(r'\{.*?\}', raw, re.DOTALL)
        if match:
            return json.loads(match.group()).get('verdict', 'PASS').upper()
    except (json.JSONDecodeError, AttributeError):
        pass
    u = raw.upper()
    if 'ESCALATE'   in u: return 'ESCALATE'
    if 'REGENERATE' in u: return 'REGENERATE'
    return 'PASS'


print(f'Scoring {len(samples)} samples with ADP-B + ADP-C v1...')
scored = []
for i, s in enumerate(samples):
    crisis_raw  = run_adp_b(s['user'])
    verdict_raw = run_adp_c_v1(s['user'], s['response'])
    verdict     = parse_verdict(verdict_raw)
    scored.append({
        'user':        s['user'],
        'response':    s['response'],
        'crisis_raw':  crisis_raw,
        'verdict_raw': verdict_raw,
        'verdict':     verdict,
    })
    if (i + 1) % 20 == 0:
        print(f'  {i+1}/{len(samples)} scored')

counts = Counter(s['verdict'] for s in scored)
print(f'\nScored {len(scored)} samples. Distribution: {dict(counts)}')


## Section 4 -- Build Training Dataset

Format scored samples as Gemma-2 chat template strings for SFTTrainer.
Stratified 90/10 split preserves verdict class balance across train and val.


In [ ]:
# -- Cell 7: Build HuggingFace Dataset ---------------------------------------
# [CONCEPT] SFTTrainer expects a Dataset with a 'text' column containing the
# fully-formatted chat template string per example. The assistant turn (verdict
# JSON) is the training target; SFTTrainer masks user/system tokens so only
# assistant tokens contribute to the cross-entropy loss.
from datasets import Dataset

# Free scoring adapters -- only the raw Gemma-2 base is needed for training.
adp_bc_model.delete_adapter('adp_b')
adp_bc_model.delete_adapter('adp_c_v1')
gc.collect()
torch.cuda.empty_cache()
print(f'VRAM after adapter cleanup: {torch.cuda.memory_allocated(0)/1024**3:.2f} GB')


def format_example(sample):
    content = f"[USER]: {sample['user']}\n[NIKKO RESPONSE]: {sample['response']}"
    verdict_json = json.dumps({'verdict': sample['verdict'], 'reason': 'pipeline output'})
    msgs = [
        {'role': 'user',      'content': content},
        {'role': 'assistant', 'content': verdict_json},
    ]
    return {'text': gemma_tok.apply_chat_template(msgs, tokenize=False), 'label': sample['verdict']}


formatted = [format_example(s) for s in scored]

train_data, val_data = train_test_split(
    formatted,
    test_size=0.1,
    stratify=[d['label'] for d in formatted],
    random_state=42,
)
train_ds = Dataset.from_list(train_data)
val_ds   = Dataset.from_list(val_data)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')
print(f"Train verdicts: {dict(Counter(train_ds['label']))}")
print(f"Val verdicts:   {dict(Counter(val_ds['label']))}")


## Section 5 -- Phase C: Train ADP-C v2

Fresh LoRA adapter on the Gemma-2 base using the same SFTTrainer / SFTConfig
pattern as Step 12. ADP-C v1 at `adp_c_final/` is not touched.


In [ ]:
# -- Cell 8: QLoRA setup -----------------------------------------------------
# [CONCEPT] get_peft_model wraps the raw base with a fresh LoRA adapter.
# Base weights are frozen (requires_grad=False); only the rank-8 LoRA delta
# matrices on q/k/v/o projections are trainable (~4.7M parameters vs ~2.6B
# base parameters). This is why QLoRA trains on an 8 GB GPU.
from trl import SFTTrainer, SFTConfig

lora_cfg = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGETS,
    bias='none',
)

# Wrap the raw gemma_base (adapters already deleted above) with a fresh LoRA.
adp_c_v2_model = get_peft_model(gemma_base, lora_cfg)
adp_c_v2_model.print_trainable_parameters()

sft_cfg = SFTConfig(
    output_dir=str(ADP_C_V2),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=0.05,
    lr_scheduler_type='cosine',
    bf16=True,
    logging_steps=5,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    max_seq_length=512,
    dataset_text_field='text',
    report_to='none',
)

trainer = SFTTrainer(
    model=adp_c_v2_model,
    args=sft_cfg,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=gemma_tok,
)
print('Trainer ready.')


In [ ]:
# -- Cell 9: Train -----------------------------------------------------------
trainer.train()
print('Training complete.')


In [ ]:
# -- Cell 10: Save adapter + smoke test --------------------------------------
# Save the best checkpoint (load_best_model_at_end=True already selected it).
trainer.save_model(str(ADP_C_V2))
gemma_tok.save_pretrained(str(ADP_C_V2))
print(f'ADP-C v2 saved to: {ADP_C_V2}')

# Smoke-test: reload from disk and run one forward pass to confirm the adapter
# is loadable. If this errors, the saved weights are corrupt.
print('\nSmoke-testing saved adapter...')
smoke_base  = AutoModelForCausalLM.from_pretrained(
    GEMMA_BASE_ID, torch_dtype=torch.bfloat16, device_map='auto'
)
smoke_model = PeftModel.from_pretrained(smoke_base, str(ADP_C_V2))
smoke_model.eval()

test_content = '[USER]: I feel really sad today.\n[NIKKO RESPONSE]: I am here for you.'
test_msgs    = [{'role': 'user', 'content': test_content}]
test_prompt  = gemma_tok.apply_chat_template(test_msgs, tokenize=False, add_generation_prompt=True)
test_inputs  = gemma_tok(test_prompt, return_tensors='pt').to(smoke_base.device)

with torch.no_grad():
    test_out = smoke_model.generate(
        **test_inputs, max_new_tokens=80, do_sample=False,
        pad_token_id=gemma_tok.eos_token_id,
    )
result = gemma_tok.decode(
    test_out[0][test_inputs['input_ids'].shape[-1]:], skip_special_tokens=True
)
print(f'Smoke test output: {result}')
